In [95]:
import os
import sys
import importlib 
import numpy as np
import matplotlib.pyplot as plot
from functools import lru_cache

from mealpy import FloatVar, PSO, GWO, MFO
from mealpy.evolutionary_based import GA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
import src.utils as utils
import src.band_segmentation as band_segmentation
importlib.reload(utils)
importlib.reload(band_segmentation)
from src.utils import load_data 
from src.band_segmentation import get_band_corr_groups



# HSI data & Segmentation

In [51]:
dataset = "IP"  # IP | PU | SA
OPT_EPOCH = 30
OPT_POP = 60
K = 2  # bands to select per correlation tier
META_METHODS = ["PSO", "GWO", "MFO", "GA"]
TEST_RATIO = 0.5  # holdout fraction used inside fitness (faster than k-fold)
FITNESS_MAX_SAMPLES = 1500  # cap labeled pixels per fitness call

In [92]:
X, y = load_data(dataset)
if dataset == 'IP': # seg-3
    regions = [
        X[:, :,   0: 36],   # bands 1-36   (36 bands)
        X[:, :,  36:102],   # bands 37-102 (66 bands)
        X[:, :, 102:200]    # bands 103-200 (98 bands)
    ]
elif dataset == 'PU': # seg-3
    regions = [
        X[:, :,  0: 37],    # bands 1-37   (37 bands) corr ~0.96
        X[:, :, 37:72],     # bands 38-72  (35 bands) corr ~0.96
        X[:, :, 72:103]     # bands 73-103 (31 bands) corr ~0.95
    ]
elif dataset == 'SA': # seg-4
    regions = [
        X[:, :,   0: 36],   # bands 1-36   (36 bands) corr ~0.91
        X[:, :,  36: 81],   # bands 37-81  (45 bands) corr ~0.90
        X[:, :,  81:106],   # bands 82-106 (25 bands) corr ~0.94
        X[:, :, 106:204],   # bands 107-204 (98 bands) corr ~0.96
    ]

corr_groups = get_band_corr_groups(X)

# Feature selection by 
#### MO(PSO, GWO, MFO, GA)

In [ ]:

mask = y > 0
y_all = y[mask] - 1
X_labeled = X[mask].astype(np.float64)

OPTIMIZERS = {
    "PSO": PSO.OriginalPSO,
    "GWO": GWO.OriginalGWO,
    "MFO": MFO.OriginalMFO,
    "GA": GA.BaseGA,
}
 
TARGET_CLF_FOR_OPTIMIZATION = 'ExtraTrees'

def get_classifier(name):
    if name == 'SVM': return SVC(kernel='rbf')
    if name == 'KNN': return KNeighborsClassifier(n_neighbors=5)
    if name == 'RF': return RandomForestClassifier(n_estimators=20, random_state=42) # Lightweight for optimizer
    if name == 'ExtraTrees': return ExtraTreesClassifier(n_estimators=20, random_state=42)
    raise ValueError("Unknown classifier name")

def optimize_group(seg_data, method='PSO'):
    n_feat = seg_data.shape[2]
    X_flat = seg_data[mask]
    X_tr, _, y_tr, _ = train_test_split(X_flat, y_all, test_size=0.2, random_state=345, stratify=y_all)
    if X_tr.shape[0] > 5000:
        idx = np.random.choice(X_tr.shape[0], 5000, replace=False)
        X_tr, y_tr = X_tr[idx], y_tr[idx]
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)

    def fitness(sol):
        binary = (np.array(sol) > 0.5).astype(int)
        n = int(np.sum(binary))
        if n < 2: return 0.0
        sel = np.where(binary == 1)[0]
        
        # === DYNAMIC CLASSIFIER LOADING ===
        clf = get_classifier(TARGET_CLF_FOR_OPTIMIZATION)
        # ==================================
        
        scores = cross_val_score(clf, X_tr[:, sel], y_tr, cv=3, scoring='accuracy')
        return scores.mean() - (0.01 * n) 

    problem = {
        'obj_func': lambda s: -fitness(s),
        'bounds': FloatVar(lb=[0]*n_feat, ub=[1]*n_feat),
        'minmax': 'min', 'log_to': None
    }
    optimizers = {'PSO': PSO.OriginalPSO, 'GWO': GWO.OriginalGWO, 'MFO': MFO.OriginalMFO, 'GA': GA.BaseGA}
    opt = optimizers[method](epoch=META_EPOCH, pop_size=META_POP)
    opt.solve(problem)
    
    best = opt.g_best.solution
    best_bin = (np.array(best) > 0.5).astype(int)
    sel_idx = np.where(best_bin == 1)[0]
    print(f'  {method} selected {len(sel_idx)} features -> indices {sel_idx}')
    return seg_data[:, :, sel_idx]


mo_results = {}

for method in META_METHODS:
    print(f"\n=== {method} ===")
    selected_per_tier = []

    for seg in  regions :  
        sel_global = optimize_group(seg, method=method)

        # selected_band = seg[sel_global]
        # selected_per_tier.append(selected_band) 
        print(f"  {method}: {len(sel_global)}")
    # mo_results[method] = np.concatenate(selected_per_tier)

print("\nSelected global band indices:")
for method, bands in mo_results.items():
    print(f"  {method}: {bands}")


=== PSO ===
  PSO selected 6 features -> indices [16 18 23 27 33 35]
  PSO: 145
  PSO selected 20 features -> indices [ 0  2  9 13 18 21 22 26 27 31 36 38 40 42 48 50 52 53 58 61]
  PSO: 145
  PSO selected 34 features -> indices [ 0  1  3  7  9 10 11 12 14 18 19 22 24 28 30 31 35 43 45 46 47 51 54 57
 61 70 74 75 81 85 91 92 94 96]
  PSO: 145

=== GWO ===
  GWO selected 11 features -> indices [ 1  7 13 14 19 20 28 30 31 33 34]
  GWO: 145
  GWO selected 15 features -> indices [ 0  3  9 22 23 32 36 41 43 45 51 52 54 62 65]
  GWO: 145
  GWO selected 22 features -> indices [ 2  4  6 17 22 26 27 31 36 42 43 46 50 53 63 65 66 72 79 83 91 93]
  GWO: 145

=== MFO ===
  MFO selected 4 features -> indices [ 7 11 26 35]
  MFO: 145
  MFO selected 5 features -> indices [ 0 15 17 33 54]
  MFO: 145
  MFO selected 9 features -> indices [15 23 48 49 51 57 70 83 89]
  MFO: 145

=== GA ===
  GA selected 8 features -> indices [ 8 11 13 16 29 30 34 35]
  GA: 145
  GA selected 19 features -> indices [ 0  2